Open and parse lists, results and original problems 

In [1]:
import pandas as pd
from typing import Dict, Any

def normalize_gt_dict(gt: Dict[str, Any],
                      key_col_name: str = "House",
                      key_out_name: str = "Key",
                      key_prefix: str = "House ") -> pd.DataFrame:
    """
    Convert a GT dictionary with structure:
        {
            "header": np.array([...]),
            "rows": np.array([
                np.array([...]),
                ...
            ])
        }
    into a tidy pandas DataFrame matching the prediction format.

    Parameters
    ----------
    gt : dict
        Ground-truth dictionary with "header" and "rows".
    key_col_name : str
        Column name in the header that holds the numeric/ordinal ID (e.g. "House").
    key_out_name : str
        Column name to output as the key (e.g. "Key").
    key_prefix : str
        Prefix for the ID values (e.g. "House ").

    Returns
    -------
    pd.DataFrame
        Tidy DataFrame with columns [Key, ...].
    """
    # Build initial DataFrame from the header and rows
    header = list(gt["header"])
    rows = [list(r) for r in gt["rows"]]
    df = pd.DataFrame(rows, columns=header)

    # Add "Key" column from the House column, then drop the original
    if key_col_name in df.columns:
        df[key_out_name] = df[key_col_name].map(lambda v: f"{key_prefix}{v}")
        df = df.drop(columns=[key_col_name])
        # Reorder so Key is first
        df = df[[key_out_name] + [c for c in df.columns if c != key_out_name]]

    return df




problem_path = "./data/grid_mode_test.parquet"

problem_df = pd.read_parquet(problem_path)
problem_df["solution"] = problem_df["solution"].apply(normalize_gt_dict)

print(problem_df.iloc[100])

id                                              lgp-test-5x3-28
size                                                        5*3
puzzle        There are 5 houses, numbered 1 to 5 from left ...
solution             Key    Name    Education Birthday
0  Ho...
created_at                           2024-07-03T21:21:29.208641
Name: 100, dtype: object


In [2]:
list_path = "./exports/all_models_fail__json.csv"

list_df = pd.read_csv(list_path)

uids = list_df["uid"].tolist()

print(uids)

['lgp-test-2x3-10', 'lgp-test-2x3-31', 'lgp-test-2x4-17', 'lgp-test-2x4-37', 'lgp-test-2x4-38', 'lgp-test-2x4-8', 'lgp-test-2x5-1', 'lgp-test-2x5-17', 'lgp-test-2x5-29', 'lgp-test-2x5-33', 'lgp-test-2x5-39', 'lgp-test-2x6-0', 'lgp-test-2x6-1', 'lgp-test-2x6-11', 'lgp-test-2x6-12', 'lgp-test-2x6-13', 'lgp-test-2x6-15', 'lgp-test-2x6-16', 'lgp-test-2x6-17', 'lgp-test-2x6-19', 'lgp-test-2x6-22', 'lgp-test-2x6-23', 'lgp-test-2x6-25', 'lgp-test-2x6-26', 'lgp-test-2x6-30', 'lgp-test-2x6-33', 'lgp-test-2x6-34', 'lgp-test-2x6-35', 'lgp-test-2x6-38', 'lgp-test-2x6-39', 'lgp-test-2x6-6', 'lgp-test-2x6-7', 'lgp-test-2x6-8', 'lgp-test-3x2-0', 'lgp-test-3x2-10', 'lgp-test-3x2-17', 'lgp-test-3x2-20', 'lgp-test-3x2-25', 'lgp-test-3x2-29', 'lgp-test-3x2-4', 'lgp-test-3x2-6', 'lgp-test-3x3-0', 'lgp-test-3x3-1', 'lgp-test-3x3-10', 'lgp-test-3x3-11', 'lgp-test-3x3-12', 'lgp-test-3x3-16', 'lgp-test-3x3-17', 'lgp-test-3x3-18', 'lgp-test-3x3-19', 'lgp-test-3x3-20', 'lgp-test-3x3-21', 'lgp-test-3x3-22', 'lgp

In [3]:
import os

results_dir = "./results_dir_parsed/zebra-grid-mini"
files_by_subdir = {}

for subdir in os.listdir(results_dir):
    subdir_path = os.path.join(results_dir, subdir)
    if os.path.isdir(subdir_path):
        files_by_subdir[subdir] = []
        for file in os.listdir(subdir_path):
            if file.endswith(".json"):
                files_by_subdir[subdir].append(results_dir + '/' + subdir + '/' + file)

print("Files by subdirectory:")
for subdir, files in files_by_subdir.items():
    print(f"{subdir}: {len(files)} files")
    for file in files:
        print(f"  - {file}")

Files by subdirectory:
xml: 6 files
  - ./results_dir_parsed/zebra-grid-mini/xml/Qwen_Qwen2.5-Math-7B-Instruct.json
  - ./results_dir_parsed/zebra-grid-mini/xml/Qwen_Qwen2.5-7B-Instruct.json
  - ./results_dir_parsed/zebra-grid-mini/xml/meta-llama_Llama-3.1-8B-Instruct.json
  - ./results_dir_parsed/zebra-grid-mini/xml/Qwen_Qwen2.5-coder-14B-Instruct.json
  - ./results_dir_parsed/zebra-grid-mini/xml/mistralai_Mistral-7B-Instruct-v0.1.json
  - ./results_dir_parsed/zebra-grid-mini/xml/Qwen_Qwen2.5-14B-Instruct.json
json: 6 files
  - ./results_dir_parsed/zebra-grid-mini/json/Qwen_Qwen2.5-Math-7B-Instruct.json
  - ./results_dir_parsed/zebra-grid-mini/json/Qwen_Qwen2.5-7B-Instruct.json
  - ./results_dir_parsed/zebra-grid-mini/json/meta-llama_Llama-3.1-8B-Instruct.json
  - ./results_dir_parsed/zebra-grid-mini/json/Qwen_Qwen2.5-coder-14B-Instruct.json
  - ./results_dir_parsed/zebra-grid-mini/json/mistralai_Mistral-7B-Instruct-v0.1.json
  - ./results_dir_parsed/zebra-grid-mini/json/Qwen_Qwen2.5-

In [12]:
import json
from typing import Dict, List, Any
import pandas as pd
from IPython.display import HTML, display
from pandas.io.formats.style import Styler


def load_json(file_path):
    with open(file_path, 'r') as file:
        try:
            return json.load(file)
        except json.JSONDecodeError:
            print(f"Error decoding JSON in file {file_path}")
            return None

def parse_results_file(file_path, key):
    results_dict = {}
    data = load_json(file_path)
    # print(len(data))
    if data:
        for item in data:
            # print(item[key])
            results_dict[item[key]] = item
    return results_dict

def extract_prediction(results):
    for result in results.values():
        predictions = [extract_last_complete_json(o) for o in result["output"]]
        if predictions[0]:
            result["prediction"] = predictions[0]["solution"]
    return results

def extract_last_complete_json(s):
    # Stack to keep track of opening and closing braces
    stack = []
    last_json_start = None
    last_json_str = None
    
    for i, char in enumerate(s):
        if char == '{':
            stack.append(i)
            if last_json_start is None:
                last_json_start = i
        elif char == '}':
            if stack:
                start = stack.pop()
                if not stack:
                    # Complete JSON object found
                    last_json_str = s[last_json_start:i+1]
                    last_json_start = None
    
    # Load the last JSON object
    if last_json_str:
        try:
            return json.loads(last_json_str.replace("\n", ""))
        except json.JSONDecodeError:
            pass
    
    return None

def dict_of_dicts_to_df(d: dict) -> pd.DataFrame:
    """
    Convert a dictionary of dictionaries into a tidy pandas DataFrame.
    The outer keys become a column ('Key'), and inner keys become columns.
    """
    df = pd.DataFrame.from_dict(d, orient='index')
    df.index.name = 'Key'
    return df.reset_index()

def align_tables(pred_df: pd.DataFrame, gt_df: pd.DataFrame):
    """
    Align two DataFrames on the union of rows/columns so we can compare cell-by-cell.
    """
    rows = sorted(set(pred_df.index).union(gt_df.index))
    cols = sorted(set(pred_df.columns).union(gt_df.columns))
    return pred_df.reindex(index=rows, columns=cols), gt_df.reindex(index=rows, columns=cols)

def diff_mask(pred_df: pd.DataFrame, gt_df: pd.DataFrame) -> pd.DataFrame:
    """
    Boolean mask where True marks a mismatch. Uses string comparison for robustness.
    """
    a = pred_df.astype("string")
    b = gt_df.astype("string")
    # Treat <NA> same way on both sides
    return (a.fillna("<NA>") != b.fillna("<NA>"))

def style_table(df: pd.DataFrame, mask: pd.DataFrame | None, title: str) -> Styler:
    """
    Build a styled table; mismatches shaded. Title shown above the table.
    Text forced to black for readability.
    """
    if "Key" in df.columns:
        cols = ["Key"] + [c for c in df.columns if c != "Key"]
        df = df[cols]
        if mask is not None:
            mask = mask[cols]

    styler = (
        df.style
          .set_caption(title)
          .set_table_styles([
              {"selector": "caption", "props": [("caption-side", "top"),
                                                ("font-weight", "bold"),
                                                ("font-size", "1.1em"),
                                                ("margin-bottom", "0.25rem"),
                                                ("color", "black")]},
              {"selector": "th", "props": [("background-color", "white"),
                                           ("color", "black"),
                                           ("font-weight", "bold"),
                                           ("border", "1px solid #ddd"),
                                           ("padding", "6px 10px")]},
              {"selector": "td", "props": [("padding", "6px 10px"),
                                           ("color", "black"),
                                           ("border", "1px solid #ddd")]},
              {"selector": "table", "props": [("border-collapse", "collapse"),
                                              ("border", "1px solid #ddd"),
                                              ("color", "black")]}
          ])
    )

    if mask is not None:
        # build a style dataframe of same shape
        styles = mask.astype(str).replace(
            {"True": "background-color: #eb5e28; color: black;",
             "False": "background-color: white; color: black;"}
        )
        styler = styler.apply(lambda _: styles, axis=None)

    return styler

def render_result(result: Dict[str, Any], index: int | None = None, data_type: str = "NULL", model: str = "Model"):
    """
    Render one result with problem + reasoning + side-by-side tables.
    Expected keys in each result:
      - "prediction": dict-of-dicts table
      - "ground_truth": dict-of-dicts table
      - "puzzle": str
      - "reasoning": str
    """
    # Extract & convert tables
    # print(result)

    pred_df = result["prediction"]
    gt_df   = result["ground_truth"]

    # Align for comparison and build mask
    # print(pred_df)
    # print(gt_df)
    pred_df_aligned, gt_df_aligned = align_tables(pred_df, gt_df)
    mask = diff_mask(pred_df_aligned, gt_df_aligned)

    # Build stylers
    pred_st = style_table(pred_df_aligned, mask, "Prediction")
    gt_st   = style_table(gt_df_aligned, mask, "Ground Truth")

    # Wrap everything in a simple flex layout
    title = f"{model} // {data_type} // {result['id']}" if result['id'] is not None else "Result"
    header_html = f"""
    <div style="border:1px solid #ddd;border-radius:10px;padding:12px;margin:10px 0;">
      <div style="font-weight:700;font-size:1.1em;margin-bottom:8px;">{title}</div>
      <div style="margin-bottom:6px;"><strong>Puzzle:</strong> {result.get('puzzle','')}</div>
      <div style="white-space:pre-wrap;background:#252422;border:1px solid #eee;padding:8px;border-radius:8px;">
        <strong>Reasoning:</strong>
        <div>{result.get('reasoning','')}</div>
      </div>
      <div style="display:flex;gap:16px;align-items:flex-start;margin-top:10px;flex-wrap:wrap;">
        <div style="flex:1;min-width:300px;">{pred_st.to_html()}</div>
        <div style="flex:1;min-width:300px;">{gt_st.to_html()}</div>
      </div>
      <div style="margin-top:8px;font-size:0.9em;color:#555;">
        <em>Cells shaded indicate mismatches between prediction and ground truth.</em>
      </div>
    </div>
    """
    display(HTML(header_html))

def render_results(ids: List[str], results_dict: Dict[str, Dict[str, Any]], data_type: str, model: str, limit: int | None = None):
    """
    Iterate through a list of IDs and render the corresponding results from results_dict.

    Parameters
    ----------
    ids : list of str
        List of result IDs to render (in order).
    results_dict : dict
        Dictionary mapping IDs -> result dicts.
    limit : int, optional
        Maximum number of results to render (useful for previews).
    """
    n = len(ids) if limit is None else min(limit, len(ids))
    for i in range(n):
        rid = ids[i]
        if rid not in results_dict:
            print(f"Warning: ID '{rid}' not found in results_dict, skipping.")
            continue
        render_result(results_dict[rid], index=i+1, data_type=data_type, model=model)

In [19]:
file = files_by_subdir['xml'][2]
type = file.split('/')[-2]
model = file.split('/')[-1]
print(model)
raw_results = parse_results_file(files_by_subdir['json'][2], key = "id")

results = extract_prediction(raw_results)

for result in results.keys():
    results[result]["ground_truth"] = problem_df[problem_df['id'] == result]['solution'].values[0]
    if results[result]["parsed"] == True:
        results[result]["prediction"] = dict_of_dicts_to_df(results[result]["prediction"])

    else:
        results[result]["prediction"] = pd.DataFrame()
        results[result]["reasoning"] = "RAW OUTPUT: " + results[result]["output"][0]
        

meta-llama_Llama-3.1-8B-Instruct.json


In [20]:
render_results(uids, results, type, model)

,Key,Drink,Name,PhoneModel
0,House 1,water,Arnold,iphone 13
1,House 2,tea,Eric,samsung galaxy s21
,Key,Drink,Name,PhoneModel
0,House 1,water,Arnold,samsung galaxy s21
1,House 2,tea,Eric,iphone 13


,Key,FavoriteSport,Hobby,Name
0,House 1,basketball,photography,Eric
1,House 2,soccer,gardening,Arnold
,Key,FavoriteSport,Hobby,Name
0,House 1,basketball,gardening,Arnold
1,House 2,soccer,photography,Eric


,Key,Animal,Cigar,Name,Nationality
0,House 1,cat,prince,Arnold,dane
1,House 2,horse,pall mall,Eric,brit
,Key,Animal,Cigar,Name,Nationality
0,House 1,horse,prince,Eric,brit
1,House 2,cat,pall mall,Arnold,dane


,Key,Hobby,Name,Occupation,Pet
0,House 1,gardening,Eric,doctor,cat
1,House 2,photography,Arnold,engineer,dog
,Key,Hobby,Name,Occupation,Pet
0,House 1,photography,Arnold,engineer,cat
1,House 2,gardening,Eric,doctor,dog


,Key,Birthday,Children,Education,Name
0,nan,nan,nan,nan,nan
1,nan,nan,nan,nan,nan
,Key,Birthday,Children,Education,Name
0,House 1,april,Bella,associate,Arnold
1,House 2,sept,Fred,high school,Eric


...

,Key,HouseStyle,Name
0,nan,nan,nan
1,nan,nan,nan
2,nan,nan,nan
3,nan,nan,nan
,Key,HouseStyle,Name
0,House 1,craftsman,Peter
1,House 2,colonial,Arnold
2,House 3,victorian,Eric
3,House 4,ranch,Alice


,Key,Food,Name
0,House 1,stew,Alice
1,House 2,spaghetti,Peter
2,House 3,pizza,Arnold
3,House 4,grilled cheese,Eric
,Key,Food,Name
0,House 1,spaghetti,Peter
1,House 2,stew,Arnold
2,House 3,pizza,Alice
3,House 4,grilled cheese,Eric


,Key,Education,Name
0,House 1,master,Eric
1,House 2,high school,Arnold
2,House 3,bachelor,Alice
3,House 4,associate,Peter
,Key,Education,Name
0,House 1,master,Eric
1,House 2,bachelor,Alice
2,House 3,high school,Peter
3,House 4,associate,Arnold


,Key,Cigar,Name
0,House 1,Prince,Alice
1,House 2,Dunhill,Eric
2,House 3,Pall Mall,Arnold
3,House 4,Blue Master,Peter
,Key,Cigar,Name
0,House 1,dunhill,Alice
1,House 2,pall mall,Arnold
2,House 3,prince,Eric
3,House 4,blue master,Peter


,Key,Height,Name
0,nan,nan,nan
1,nan,nan,nan
2,nan,nan,nan
3,nan,nan,nan
,Key,Height,Name
0,House 1,short,Alice
1,House 2,average,Eric
2,House 3,tall,Peter
3,House 4,very short,Arnold
